# PathSnap on Google Colab (GPU)

Runs the PathSnap pipeline on a Colab **T4 GPU**: preprocessing, a GPU-vs-CPU speed check, and the
**Bayesian hyperparameter search for XGBoost on the GPU** (`device="cuda"`). It can also retrain any model.
It runs the same code as the laptops, uploaded by hand: no GitHub access needed.

**Before you start**
1. On a laptop with the repo: `python -m pathsnap.colab_bundle` → makes `colab/pathsnap_colab.zip`
   (code + raw data, ~30 MB).
2. In Colab: *Runtime → Change runtime type → T4 GPU*.
3. Upload `pathsnap_colab.zip` with the **Files** panel (folder icon on the left, upload button), or run the
   upload cell below. Alternatively put it in Google Drive and set `USE_DRIVE = True`.

Only XGBoost uses the GPU. The other models (scikit-learn, LightGBM CPU build) run on Colab's 2 CPU cores.
GPU and CPU XGBoost give close but not bit-identical results, so GPU trials go into their own Optuna study
(`xgb_top1_gpu`) instead of being mixed with the laptop study.

## 1. Settings

In [ ]:
BUNDLE    = "pathsnap_colab.zip"
USE_DRIVE = False                          # True: read the zip from / save outputs to Google Drive
DRIVE_DIR = "/content/drive/MyDrive/PathSnap"
STUDY     = "xgb_top1_gpu"                 # "xgb_top1" continues the laptop (CPU) study inside the zip
N_TRIALS  = 50                             # total trials in the study (resumes if some already exist)

## 2. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, multiprocessing
print("CPU cores:", multiprocessing.cpu_count())

## 3. Load the bundle

Looks for the zip in `/content` (Files-panel upload) or on Drive; otherwise opens an upload dialog.

In [ ]:
import pathlib, zipfile
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
candidates = [pathlib.Path("/content") / BUNDLE, pathlib.Path(DRIVE_DIR) / BUNDLE]
bundle = next((p for p in candidates if p.exists()), None)
if bundle is None:
    from google.colab import files
    print(f"{BUNDLE} not found - choose it in the dialog")
    uploaded = files.upload()
    bundle = pathlib.Path("/content") / next(iter(uploaded))
print("bundle:", bundle, f"({bundle.stat().st_size / 1e6:.1f} MB)")

WORK = pathlib.Path("/content/work")
if not (WORK / "pathsnap").exists():
    with zipfile.ZipFile(bundle) as z:
        z.extractall(WORK)
REPO_DIR = WORK / "pathsnap"
DATA_DIR = WORK / "Student folder"
assert (REPO_DIR / "pathsnap" / "config.py").exists() and (DATA_DIR / "train").is_dir(), "unexpected zip layout"
%cd {REPO_DIR}

## 4. Install dependencies

`requirements.txt` pins `xgboost-cpu` (laptops); Colab gets the full `xgboost` wheel of the same version, which includes CUDA.

In [ ]:
import re, subprocess
reqs = [l.strip() for l in open("requirements.txt") if l.strip() and not l.startswith("#")]
xgb_version = next(l.split("==")[1] for l in reqs if l.startswith("xgboost-cpu"))
others = [l for l in reqs if not l.startswith("xgboost-cpu")]
open("/tmp/req_colab.txt", "w").write("\n".join(others + [f"xgboost=={xgb_version}"]))
r = subprocess.run(["pip", "install", "-q", "-r", "/tmp/req_colab.txt"], capture_output=True, text=True)
if r.returncode != 0:
    # exact pins unavailable for Colab's Python: fall back to the same packages unpinned
    print("pinned install failed, installing unpinned:", r.stderr[-500:])
    names = [re.split(r"[=<>]", l)[0] for l in others] + [f"xgboost=={xgb_version}"]
    subprocess.run(["pip", "install", "-q", *names], check=True)
import xgboost, pandas, sklearn
print("xgboost", xgboost.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)

## 5. Environment

In [ ]:
os.environ["PATHSNAP_DATA"] = str(DATA_DIR)
os.environ["PATHSNAP_JOBS"] = str(multiprocessing.cpu_count())
os.environ["PATHSNAP_XGB_DEVICE"] = "cuda"
os.environ["PATHSNAP_STUDY"] = STUDY
import sys
sys.path.insert(0, str(REPO_DIR))
!python -m pathsnap.reproduce --check

## 6. Preprocessing (builds `data/processed/`, ~5-8 min on Colab's 2 cores)

In [ ]:
from pathsnap.reproduce import ensure_processed
ensure_processed()

## 7. GPU vs CPU speed check (one XGBoost fit on one cross-validation fold)

In [ ]:
import time
from sklearn.model_selection import GroupKFold
from xgboost import XGBClassifier
from pathsnap.models.evaluate import load_candidates, evaluate
from pathsnap.features.candidates import FEATURE_COLUMNS
from pathsnap.hpo.xgb_optuna import BASELINE

train = load_candidates("train")
tr, va = next(GroupKFold(n_splits=5).split(train, groups=train["trip_id"]))
X, y = train[FEATURE_COLUMNS].to_numpy(), train["label"].to_numpy()
for device in ["cuda", "cpu"]:
    t0 = time.perf_counter()
    m = XGBClassifier(**BASELINE, tree_method="hist", device=device, n_jobs=multiprocessing.cpu_count(),
                      random_state=42, eval_metric="logloss").fit(X[tr], y[tr])
    fit_s = time.perf_counter() - t0
    top1 = evaluate(train.iloc[va], m.predict_proba(X[va])[:, 1])["top1_accuracy"]
    print(f"{device:4s}: fit {fit_s:6.1f} s | fold-0 top-1 {top1:.4f}")

## 8. Bayesian search on the GPU

Same objective, search space, folds, sampler and pruner as the laptop run (`pathsnap/hpo/xgb_optuna.py`). The study is saved after every trial in `results/hpo/xgb_study.db`; if the session disconnects, re-upload the outputs zip's `xgb_study.db` into `results/hpo/` and rerun this cell to resume.

In [ ]:
!python -m pathsnap.hpo.xgb_optuna --trials {N_TRIALS}

## 9. Refit the best configuration and compare with the untuned model

The refit goes through the same harness as every other model (5-fold CV + student test). The saved final model is switched to CPU inference so the API laptop can load it.

In [ ]:
assert os.path.exists("results/models/xgb/metrics.json"), "untuned XGBoost results missing from the bundle"
!python -m pathsnap.hpo.xgb_optuna --refit
!python -m pathsnap.viz.hpo_plots

In [ ]:
from IPython.display import Image, display
for f in ["optimisation_history.png", "param_importance.png", "tuned_vs_untuned.png"]:
    display(Image(filename=f"results/hpo/plots/{f}"))

## 10. Download the outputs

Colab storage is wiped when the session ends. This zips the search record, plots, tuned-model results and the final model, then downloads it. Unzip it inside the repo on the laptop (paths are relative to the repo root).

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M")
out = pathlib.Path(f"/content/pathsnap_outputs_{STUDY}_{stamp}.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["results/hpo", "results/models/xgb_tuned"]:
        for f in pathlib.Path(folder).rglob("*"):
            if f.is_file():
                z.write(f, f)
    z.write("results/models/comparison.csv", "results/models/comparison.csv")
    for f in ["artifacts/final_model.joblib", "artifacts/final_model.json"]:
        if os.path.exists(f):
            z.write(f, f)
print("outputs:", out, f"({out.stat().st_size / 1e6:.1f} MB)")
if USE_DRIVE:
    import shutil
    shutil.copy(out, pathlib.Path(DRIVE_DIR) / out.name)
    print("copied to Drive")
from google.colab import files
files.download(str(out))

## Training other models here (optional)

Any registered model can be retrained with the same harness; only XGBoost uses the GPU.

```python
!python -m pathsnap.models.train --list
!python -m pathsnap.models.train lgbm
```